# UzTerm AI — ko'p tilli terminlar: LaBSE va Vikipediya (Google Colab)

Dissertatsiyaning **3.3-bo'limi** uchun ikki ish shu yerda bajariladi, chunki ular internet va
(tavsiya etiladi) GPU talab qiladi:

1. **Semantik moslashtirish.** Ko'p tilli LaBSE embedding modeli bilan ruscha/inglizcha termindan
   oʻzbekcha ekvivalentni topish va qoidaviy kognat modeli bilan solishtirish
   (`app.multilingual.experiment --encoder`). Natija: `reports/multilingual.json`.
2. **Vikipediya ekvivalentlari.** Lug'atdagi terminlar uchun oʻzbekcha Vikipediya maqolalarining
   ruscha va inglizcha nomlari (`langlinks`). Natija **tekshirilmagan nomzodlar**:
   `data/multilingual/wikipedia-langlinks.csv`.

Runtime → Change runtime type → **T4 GPU** (CPU'da ham ishlaydi, sekinroq).

## 1. Repozitoriyni yuklash

Repo **private** bo'lsa, GitHub token kerak (Fine-grained, faqat shu repo, *Contents: Read-only*). Public bo'lsa, bo'sh qoldiring.

In [ ]:
import getpass, os, subprocess

REPO = "Mirzokhidbek/graduatemdi"
BRANCH = "main"

token = getpass.getpass("GitHub token (public repo bo'lsa bo'sh qoldiring): ").strip()
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("/content/graduatemdi"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/graduatemdi"], check=True)
del token, url
%cd /content/graduatemdi
!git log --oneline -1

## 2. Kutubxonalar

In [ ]:
!pip install -q sentence-transformers pydantic
import os, sys
os.environ["PYTHONPATH"] = "backend"
sys.path.insert(0, "backend")

## 3. Kognat, semantik (LaBSE) va gibrid modellarni solishtirish

`--weight` — gibrid modelda embedding og'irligi (0.5 standart).

In [ ]:
!PYTHONPATH=backend python -m app.multilingual.experiment --encoder --weight 0.5

## 4. Vikipediya: lug'atdagi terminlarning ruscha va inglizcha ekvivalentlari

Natija avtomatik tasdiqlanmaydi: har bir juftni mutaxassis ko'rib chiqadi (`status = needs_review`).

In [ ]:
import csv
from app.database.term_store import load_terms
from app.multilingual.sources import fetch_langlinks

entries = load_terms("data/terms.json")
titles = [e.term[:1].upper() + e.term[1:] for e in entries]
links = fetch_langlinks(titles)
rows = [{"uz": t.lower(), "ru": v.get("ru", ""), "en": v.get("en", ""), "source": "uz.wikipedia langlinks",
         "status": "needs_review"} for t, v in links.items()]
os.makedirs("data/multilingual", exist_ok=True)
with open("data/multilingual/wikipedia-langlinks.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["uz", "ru", "en", "source", "status"]); w.writeheader(); w.writerows(rows)
print(f"{len(rows)} / {len(titles)} ta termin uchun Vikipediya maqolasi topildi")
rows[:10]

## 5. Natijalarni yuklab olish

Fayllarni repozitoriyga qo'shing: `reports/multilingual.json`, `data/multilingual/wikipedia-langlinks.csv`.

In [ ]:
from google.colab import files
files.download("reports/multilingual.json")
files.download("data/multilingual/wikipedia-langlinks.csv")